# ETF V7：JQ-like 执行路径一致性评估

目的：修正 V6 的核心评估缺陷。V6 的 close-to-close proxy 只能说明信号方向，不能代表 JoinQuant 周调仓 `09:35` 的真实执行路径。本 notebook 不新增模型，不重新讨论行业去重，不继续调参；只把已有 V6 score panel 放进更贴近回测的组合模拟器。

核心假设：如果离线评估器按 JoinQuant 的推理/交易路径模拟，模型排序应当和真实 JoinQuant 回测大体同向。若仍然相反，优先修评估器或回测执行，而不是继续调模型。

输入：`etf_ml_v6_anchor_ml_filter_dedup_governance_outputs/etf_ml_v6_score_panel.csv`。

输出：
- `etf_ml_v7_jq_like_weekly_equity.csv`
- `etf_ml_v7_jq_like_targets.csv`
- `etf_ml_v7_jq_like_summary.csv`
- `etf_ml_v7_parity_notes.csv`


In [ ]:
# =========================
# 0. Imports and config
# =========================
try:
    from jqdata import *
except Exception as err:
    print("JoinQuant import unavailable. Price rebuild cells require JoinQuant runtime. err=", err)

import os
import math
import datetime
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SOURCE_V6_OUT_DIR = "etf_ml_v6_anchor_ml_filter_dedup_governance_outputs"
OUT_DIR = "etf_ml_v7_jq_like_execution_parity_outputs"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

SCORE_CSV = os.path.join(SOURCE_V6_OUT_DIR, "etf_ml_v6_score_panel.csv")
MODEL_MANIFEST_CSV = os.path.join(SOURCE_V6_OUT_DIR, "etf_ml_v6_model_manifest.csv")

EQUITY_CSV = os.path.join(OUT_DIR, "etf_ml_v7_jq_like_weekly_equity.csv")
TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v7_jq_like_targets.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v7_jq_like_summary.csv")
NOTES_CSV = os.path.join(OUT_DIR, "etf_ml_v7_parity_notes.csv")

# Keep this aligned with jq_backtest_etf_ml_weekly_lgb_v3.py.
INITIAL_CASH = 100000.0
TOP_N_LIST = [1, 3]
PRICE_FIELD = "open"     # approximate JoinQuant 09:35 rebalance. Use open as conservative reproducible proxy.
MIN_LOT = 100
SLIPPAGE_RATE = 0.001     # FixedSlippage(0.001) approximate; modeled as percentage haircut here.
COMMISSION_RATE = 0.0001
MIN_COMMISSION = 0.0
RISK_FREE_WEEKLY_RET = 0.0

# If True, target selection skips ETFs whose equal-weight sleeve cannot buy at least 100 shares.
CAPITAL_AWARE_TARGETS = True

print("score csv:", SCORE_CSV)
print("out dir:", OUT_DIR)


## 1. Load V6 score panel

V7 必须读取完整 score panel，因为我们需要从每周的全候选 ETF 中重新选 top1/top3，并在买不起 100 股时顺延候选。如果只有 `weekly_portfolio_proxy.csv`，那只能复盘已选结果，不能验证选股执行一致性。


In [ ]:
# =========================
# 1. Load score panel
# =========================
def require_file(path, hint):
    if not os.path.exists(path):
        raise IOError("required file not found: " + path + "\n" + hint)
    return path

require_file(
    SCORE_CSV,
    "Run ETF_V6_anchor_ml_filter_dedup_governance实验.ipynb in JoinQuant first, "
    "or copy its full output directory here. Downloads proxy files are not enough for V7."
)

score_df = pd.read_csv(SCORE_CSV)
for c in ["feature_date", "rebalance_date", "next_date"]:
    if c in score_df.columns:
        score_df[c] = pd.to_datetime(score_df[c], errors="coerce")

if "code" not in score_df.columns or "score" not in score_df.columns or "feature_date" not in score_df.columns:
    raise ValueError("score panel must contain code, score, feature_date")

score_df = score_df.replace([np.inf, -np.inf], np.nan)
score_df = score_df.dropna(subset=["code", "score", "feature_date"]).copy()
score_df["code"] = score_df["code"].astype(str)

print("score shape:", score_df.shape)
print("date range:")
print(score_df[["feature_date"]].agg(["min", "max"]))
print("models:", score_df["model_name"].dropna().nunique() if "model_name" in score_df.columns else "missing")
print(score_df.head(3).to_string())


## 2. Build executable rebalance calendar and price table

信号日期是 `feature_date`，真实回测在下一交易日/周一 `09:35` 附近交易。这里用下一交易日 `open` 近似。这个近似仍不是逐分钟成交，但比 V6 的 `feature_date close -> future close` 更接近回测。


In [ ]:
# =========================
# 2. Calendar and price helpers
# =========================
def to_date_str(x):
    return pd.Timestamp(x).strftime("%Y-%m-%d")


def get_next_trade_day_safe(date):
    try:
        days = list(get_trade_days(start_date=to_date_str(date), count=2))
    except NameError:
        raise RuntimeError("JoinQuant get_trade_days is required for V7 JQ-like simulator")
    if len(days) == 0:
        return pd.NaT
    d0 = pd.Timestamp(days[0])
    if d0.date() > pd.Timestamp(date).date():
        return d0
    if len(days) >= 2:
        return pd.Timestamp(days[1])
    return pd.NaT


def add_entry_dates(df):
    dates = sorted(pd.to_datetime(df["feature_date"].dropna().unique()))
    mp = {}
    for d in tqdm(dates, desc="entry dates"):
        mp[pd.Timestamp(d)] = get_next_trade_day_safe(d)
    out = df.copy()
    out["entry_date"] = out["feature_date"].map(mp)
    return out

score_df = add_entry_dates(score_df)
score_df = score_df.dropna(subset=["entry_date"]).copy()
print("entry date range:", score_df["entry_date"].min(), score_df["entry_date"].max())


In [ ]:
# =========================
# 3. Fetch open prices for candidates
# =========================
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def fetch_price_table(codes, start_date, end_date, chunk_size=120):
    all_parts = []
    fields = [PRICE_FIELD, "close"]
    for code_chunk in tqdm(list(chunks(list(codes), chunk_size)), desc="fetch price"):
        try:
            px = get_price(
                code_chunk,
                start_date=to_date_str(start_date),
                end_date=to_date_str(end_date),
                frequency="daily",
                fields=fields,
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except NameError:
            raise RuntimeError("JoinQuant get_price is required for V7 JQ-like simulator")
        except Exception as err:
            print("price fetch failed", code_chunk[:3], err)
            px = None
        if px is None or len(px) == 0:
            continue
        if "time" not in px.columns or "code" not in px.columns:
            continue
        px["date"] = pd.to_datetime(px["time"]).dt.normalize()
        if PRICE_FIELD not in px.columns:
            px[PRICE_FIELD] = px["close"]
        keep = px[["date", "code", PRICE_FIELD, "close"]].copy()
        keep["trade_price"] = pd.to_numeric(keep[PRICE_FIELD], errors="coerce")
        keep.loc[keep["trade_price"].isnull(), "trade_price"] = pd.to_numeric(keep["close"], errors="coerce")
        all_parts.append(keep[["date", "code", "trade_price"]])
    if len(all_parts) == 0:
        raise RuntimeError("empty price table")
    out = pd.concat(all_parts, ignore_index=True, sort=False)
    out = out.dropna(subset=["date", "code", "trade_price"])
    out = out[out["trade_price"] > 0].copy()
    return out

all_codes = sorted(score_df["code"].dropna().astype(str).unique())
min_entry = score_df["entry_date"].min()
max_entry = score_df["entry_date"].max() + pd.Timedelta(days=10)
price_df = fetch_price_table(all_codes, min_entry, max_entry)
price_map = {}
for _, row in price_df.iterrows():
    price_map[(str(row["code"]), pd.Timestamp(row["date"]).normalize())] = float(row["trade_price"])
print("price rows:", price_df.shape)
print(price_df.head(3).to_string())


## 3. JQ-like portfolio simulator

模拟规则：

1. 每个 `feature_date` 用对应模型分数排序。
2. 入场价使用下一交易日 `open` 近似 `09:35`。
3. 选 topN 时先检查等权金额是否买得起 100 股；买不起则顺延候选。
4. 先卖出不在目标中的持仓，再按等权目标买入/调仓。
5. 不足 100 股的小额调仓跳过。
6. 记录现金、持仓市值、交易费用、跳过原因和目标列表。


In [ ]:
# =========================
# 4. Simulator helpers
# =========================
def get_price_from_map(code, date):
    return price_map.get((str(code), pd.Timestamp(date).normalize()), np.nan)


def floor_lot_shares(value, price):
    if pd.isnull(price) or price <= 0 or value <= 0:
        return 0
    return int(math.floor(float(value) / float(price) / float(MIN_LOT)) * MIN_LOT)


def trade_cost(value):
    if value <= 0:
        return 0.0
    return max(MIN_COMMISSION, abs(float(value)) * COMMISSION_RATE)


def mark_to_market(holdings, date):
    value = 0.0
    bad = []
    for code, shares in holdings.items():
        if shares <= 0:
            continue
        px = get_price_from_map(code, date)
        if pd.isnull(px) or px <= 0:
            bad.append(code)
            continue
        value += float(shares) * float(px)
    return value, bad


def select_targets_for_week(month_df, topn, equity, date):
    sorted_df = month_df.sort_values("score", ascending=False).copy()
    raw_top = list(sorted_df.head(topn)["code"].astype(str))
    target_value = float(equity) / float(max(1, topn))
    selected = []
    skipped = []
    for _, row in sorted_df.iterrows():
        code = str(row["code"])
        px = get_price_from_map(code, date)
        if pd.isnull(px) or px <= 0:
            skipped.append((code, "no_price"))
            continue
        min_value = px * MIN_LOT
        if CAPITAL_AWARE_TARGETS and target_value * 0.98 < min_value:
            skipped.append((code, "min_lot"))
            continue
        selected.append(code)
        if len(selected) >= topn:
            break
    return selected, raw_top, skipped


def normalize_group_key(keys):
    if isinstance(keys, tuple):
        return keys
    return (keys,)


def calc_drawdown(nav):
    s = pd.Series(nav).astype(float)
    if len(s) == 0:
        return np.nan
    peak = s.cummax()
    dd = s / peak - 1.0
    return float(dd.min())


def summarize_equity(gdf):
    gdf = gdf.sort_values("entry_date")
    if gdf.empty:
        return {}
    start_nav = float(gdf.iloc[0]["nav_before_trade"])
    end_nav = float(gdf.iloc[-1]["nav_after_trade"])
    rets = pd.to_numeric(gdf["period_ret"], errors="coerce").dropna()
    return {
        "periods": int(len(gdf)),
        "start_date": gdf.iloc[0]["entry_date"],
        "end_date": gdf.iloc[-1]["entry_date"],
        "cum_ret": end_nav / start_nav - 1.0 if start_nav > 0 else np.nan,
        "max_drawdown": calc_drawdown(gdf["nav_after_trade"].values),
        "mean_period_ret": float(rets.mean()) if len(rets) else np.nan,
        "period_sharpe": float(rets.mean() / rets.std()) if len(rets) > 1 and rets.std() > 0 else np.nan,
        "win_rate": float((rets > 0).mean()) if len(rets) else np.nan,
        "avg_turnover": float(pd.to_numeric(gdf["turnover"], errors="coerce").mean()),
        "order_skip_weeks": int((pd.to_numeric(gdf["skip_count"], errors="coerce") > 0).sum()),
    }


In [ ]:
# =========================
# 5. Run one model/topN simulation
# =========================
def run_jq_like_sim(months_df, group_info, topn):
    cash = float(INITIAL_CASH)
    holdings = {}
    equity_rows = []
    target_rows = []
    last_nav = float(INITIAL_CASH)

    months = sorted(pd.to_datetime(months_df["feature_date"].dropna().unique()))
    for feature_date in months:
        mdf = months_df[months_df["feature_date"] == feature_date].copy()
        if mdf.empty:
            continue
        entry_date = pd.Timestamp(mdf["entry_date"].iloc[0]).normalize()
        holding_value_before, bad_marks = mark_to_market(holdings, entry_date)
        nav_before = cash + holding_value_before
        selected, raw_top, skipped = select_targets_for_week(mdf, topn, nav_before, entry_date)

        turnover_value = 0.0
        fees = 0.0
        trade_notes = []

        # Sell holdings that are no longer selected.
        for code in list(holdings.keys()):
            shares = int(holdings.get(code, 0))
            if shares <= 0:
                holdings.pop(code, None)
                continue
            if code in selected:
                continue
            px = get_price_from_map(code, entry_date)
            if pd.isnull(px) or px <= 0:
                trade_notes.append("sell_no_price:" + code)
                continue
            gross = shares * px
            slip = gross * SLIPPAGE_RATE
            fee = trade_cost(gross)
            cash += gross - slip - fee
            turnover_value += gross
            fees += fee + slip
            holdings.pop(code, None)

        # Recompute after sells, then rebalance selected basket.
        holding_value_mid, _ = mark_to_market(holdings, entry_date)
        nav_mid = cash + holding_value_mid
        target_value = nav_mid / float(max(1, len(selected))) if len(selected) else 0.0

        for code in selected:
            px = get_price_from_map(code, entry_date)
            if pd.isnull(px) or px <= 0:
                trade_notes.append("buy_no_price:" + code)
                continue
            current_shares = int(holdings.get(code, 0))
            current_value = current_shares * px
            diff_value = target_value - current_value
            if abs(diff_value) < px * MIN_LOT:
                continue
            if diff_value > 0:
                buy_value_budget = min(diff_value, cash)
                shares = floor_lot_shares(buy_value_budget / (1.0 + SLIPPAGE_RATE + COMMISSION_RATE), px)
                if shares < MIN_LOT:
                    trade_notes.append("buy_small:" + code)
                    continue
                gross = shares * px
                slip = gross * SLIPPAGE_RATE
                fee = trade_cost(gross)
                total_cash = gross + slip + fee
                if total_cash > cash:
                    trade_notes.append("buy_cash:" + code)
                    continue
                cash -= total_cash
                holdings[code] = current_shares + shares
                turnover_value += gross
                fees += fee + slip
            else:
                sell_shares = floor_lot_shares(-diff_value, px)
                sell_shares = min(sell_shares, current_shares)
                if sell_shares < MIN_LOT:
                    continue
                gross = sell_shares * px
                slip = gross * SLIPPAGE_RATE
                fee = trade_cost(gross)
                cash += gross - slip - fee
                new_shares = current_shares - sell_shares
                if new_shares > 0:
                    holdings[code] = new_shares
                else:
                    holdings.pop(code, None)
                turnover_value += gross
                fees += fee + slip

        holding_value_after, bad_after = mark_to_market(holdings, entry_date)
        nav_after = cash + holding_value_after
        period_ret = nav_after / last_nav - 1.0 if last_nav > 0 else np.nan
        turnover = turnover_value / nav_before if nav_before > 0 else np.nan
        skip_count = len(skipped) + len(trade_notes) + len(bad_marks) + len(bad_after)

        rec = {
            "feature_date": feature_date,
            "entry_date": entry_date,
            "topn": topn,
            "raw_top": ",".join(raw_top),
            "targets": ",".join(selected),
            "holdings": ",".join([k + ":" + str(int(v)) for k, v in sorted(holdings.items()) if v > 0]),
            "cash": cash,
            "holding_value": holding_value_after,
            "nav_before_trade": nav_before,
            "nav_after_trade": nav_after,
            "period_ret": period_ret,
            "turnover": turnover,
            "fees_and_slippage": fees,
            "skip_count": skip_count,
            "skip_notes": ";".join([a + ":" + b for a, b in skipped] + trade_notes + ["bad_mark:" + x for x in bad_marks + bad_after]),
        }
        for k, v in group_info.items():
            rec[k] = v
        equity_rows.append(rec)

        rank_map = {}
        tmp = mdf.sort_values("score", ascending=False).copy()
        for idx, code in enumerate(tmp["code"].astype(str).tolist()):
            rank_map[code] = idx + 1
        for code in selected:
            tr = {
                "feature_date": feature_date,
                "entry_date": entry_date,
                "code": code,
                "topn": topn,
                "rank": rank_map.get(code, np.nan),
                "price": get_price_from_map(code, entry_date),
            }
            for k, v in group_info.items():
                tr[k] = v
            target_rows.append(tr)

        last_nav = nav_after

    return pd.DataFrame(equity_rows), pd.DataFrame(target_rows)


In [ ]:
# =========================
# 6. Run all model/topN simulations
# =========================
GROUP_COLS = [c for c in ["model_name", "score_family", "target_name", "train_tag"] if c in score_df.columns]
if "model_name" not in GROUP_COLS:
    raise ValueError("score panel missing model_name; cannot group simulations")

all_equity = []
all_targets = []
groups = list(score_df.groupby(GROUP_COLS))
for keys, gdf in tqdm(groups, desc="model groups"):
    keys = normalize_group_key(keys)
    group_info = {}
    for i, col in enumerate(GROUP_COLS):
        group_info[col] = keys[i]
    for topn in TOP_N_LIST:
        eq, tg = run_jq_like_sim(gdf, group_info, topn)
        if not eq.empty:
            all_equity.append(eq)
        if not tg.empty:
            all_targets.append(tg)
        del eq, tg

equity_df = pd.concat(all_equity, ignore_index=True, sort=False) if len(all_equity) else pd.DataFrame()
targets_df = pd.concat(all_targets, ignore_index=True, sort=False) if len(all_targets) else pd.DataFrame()
print("equity shape:", equity_df.shape)
print("targets shape:", targets_df.shape)
print(equity_df.head(3).to_string() if not equity_df.empty else "EMPTY")


## 4. Summaries and parity notes

优先看 `cum_ret`、`max_drawdown`、`avg_turnover`、`order_skip_weeks`，再和真实 JoinQuant 回测排序对比。如果这个排序和真实回测仍然反着来，说明仍缺关键执行细节，不要继续根据 proxy 调模型。


In [ ]:
# =========================
# 7. Summary
# =========================
summary_rows = []
if not equity_df.empty:
    group_cols = [c for c in GROUP_COLS + ["topn"] if c in equity_df.columns]
    for keys, gdf in equity_df.groupby(group_cols):
        keys = normalize_group_key(keys)
        rec = {}
        for i, col in enumerate(group_cols):
            rec[col] = keys[i]
        rec.update(summarize_equity(gdf))
        summary_rows.append(rec)
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["topn", "cum_ret"], ascending=[True, False])
print(summary_df.to_string(index=False) if not summary_df.empty else "EMPTY")

notes = [{
    "item": "evaluation_rule",
    "value": "V7 uses next trade day open as 09:35 proxy, capital-aware topN selection, 100-share lot, cash, fees, slippage, and small-adjust skip.",
}, {
    "item": "not_equivalent_to_real_backtest",
    "value": "Still not a minute-level JoinQuant fill simulator. Real JoinQuant backtest remains final judge.",
}, {
    "item": "acceptance_rule",
    "value": "If V7 JQ-like ranking and JoinQuant backtest ranking disagree, fix simulator/execution before model tuning.",
}]
notes_df = pd.DataFrame(notes)


In [ ]:
# =========================
# 8. Save outputs
# =========================
equity_df.to_csv(EQUITY_CSV, index=False)
targets_df.to_csv(TARGETS_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
notes_df.to_csv(NOTES_CSV, index=False)

for p in [EQUITY_CSV, TARGETS_CSV, SUMMARY_CSV, NOTES_CSV]:
    print("saved:", p, os.path.exists(p))


## Self Review

- 本实验不新增模型，只修评估路径。
- 训练特征仍来自 V6 score panel，本 notebook 不重新训练，也不改变分数。
- 资金约束和 100 股约束会改变 topN，因此 target list 可能不同于 V6 close-to-close proxy，这是预期行为。
- 如果只有 Downloads 的 `weekly_portfolio_proxy.csv`，不能跑 V7；必须有完整 `etf_ml_v6_score_panel.csv`。
- 下一步：用 V7 summary 选择少数候选，再跑 `jq_backtest_etf_ml_weekly_lgb_v3.py` 真回测验证排序是否同向。
